In [ ]:
# jalankan ini di terminal agar gambar bisa dibaca dan diubah menjadi teks
# docker exec -u 0 -it jupyter sh -c "apt-get update && apt-get install -y tesseract-ocr tesseract-ocr-ind libgl1"

In [2]:
!pip install pytesseract opencv-python-headless

In [4]:
import cv2
import pytesseract
import re
import pandas as pd

def extract_receipt_items(image_path):
    img = cv2.imread(image_path)
    config = '--psm 6'
    text_content = pytesseract.image_to_string(img, lang='ind+eng', config=config)
    
    lines = [line.strip() for line in text_content.split('\n') if line.strip()]
    items = []
    
    i = 0
    while i < len(lines):
        match_title = re.match(r'^\d+\.\s*(.+)', lines[i])
        if match_title:
            nama_produk = match_title.group(1).strip()
            
            if not nama_produk.replace('.', '').strip():
                i += 1
                continue
            
            detail_lines = []
            j = i + 1
            while j < len(lines) and not re.match(r'^\d+\.', lines[j]) and not re.search(r'Total', lines[j], re.IGNORECASE):
                detail_lines.append(lines[j])
                j += 1
            
            detail_text = " ".join(detail_lines)
            
            qty = None
            match_qty = re.search(r'^\s*(.*?)\s*x', detail_text, re.IGNORECASE)
            if match_qty:
                qty = match_qty.group(1).strip()
            
            harga = None
            match_rp = re.search(r'(?:Rp|RP|rp|Pp|Bp|P)\s*([\d\.,\s]+)', detail_text, re.IGNORECASE)
            if match_rp:
                digits = re.sub(r'\D', '', match_rp.group(1))
                if digits:
                    harga = int(digits)
            if harga is None:
                all_numbers = re.findall(r'[\d\.,]+', detail_text)
                if all_numbers:
                    last_num_clean = re.sub(r'\D', '', all_numbers[-1])
                    if last_num_clean:
                        harga = int(last_num_clean)
            items.append({
                "nama_produk": nama_produk,
                "qty": qty,
                "harga": harga
            })
            i = j - 1
        i += 1
    return pd.DataFrame(items)

df = extract_receipt_items('struk.png')
df = df[df['harga'] > 100].reset_index(drop=True)

print(df)

           nama_produk      qty    harga
0       Indomie Goreng  1 lusin  36000.0
1  Belfood Sosis Bakar        1  27000.0
